# Benchmarking convergence for TMDD model

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

%load_ext autoreload
%autoreload 2

from vpop_calibration import *


In [ ]:
# Make sure to run  `generate_synthetic_data.R` first
train_df = pd.read_csv("./data/gp_training.csv")
# Make sure this is aligned with the available observed data
df = pd.read_csv(f"./data/obs_data_100.csv")
obs_df = df[["id", "protocol_arm", "time", "DV"]]
obs_df = obs_df.loc[~obs_df["DV"].isna()]
obs_df = obs_df.rename(columns={"DV": "value"})
obs_df["output_name"] = "DV"
patients_df = obs_df[["id", "protocol_arm"]].drop_duplicates()

In [ ]:
def train_model(input_df: pd.DataFrame) -> GP:
    descriptors = ["R0", "k_eL", "Vc"]
    gp = GP(
        input_df,
        descriptors=descriptors + ["time"],
        deep_kernel=False,
        nb_inducing_points=100,
        min_delta=0.01,
        nb_training_iter=500,
        log_inputs=descriptors,
        log_outputs=[],
        learning_rate=0.1,
        lr_decay=0.995,
        plot_frame_rate=10,
    )
    gp.train()

    return gp

In [ ]:
gp = train_model(train_df)

In [ ]:
gp.plot_obs_vs_predicted("training", (3, 3), logScale=[False])

In [ ]:
# def run_saem():
structural_gp = StructuralGp(gp)
# NLME model parameters
init_log_MI = {}
init_log_PDU = {
    "k_eL": {"mean": 0.0, "sd": 0.5},
    "R0": {"mean": 0.0, "sd": 0.5},
    "Vc": {"mean": 0.0, "sd": 0.5},
}

error_model_type = "additive"
init_res_var = [0.5]

nlme_surrogate = NlmeModel(
    structural_gp,
    patients_df,
    init_log_MI,
    init_log_PDU,
    init_res_var,
    None,
    error_model_type,
    num_chains=5,
    # constraints=structural_gp.training_ranges,
)

optimizer = PySaem(
    nlme_surrogate,
    obs_df,
    nb_phase1_iterations=100,
    nb_phase2_iterations=100,
    mcmc_nb_transitions=5,
    mcmc_first_burn_in=10,
    verbose=False,
    live_plot=True,
)
optimizer.run()
nlme_surrogate.population_betas

In [ ]:
nlme_surrogate.compute_ebe(samples_cond_dist=100, override=True)
map_df = nlme_surrogate.map_estimates_descriptors()
map_df.to_csv("./outputs/ebe_pysaem.csv", index=False)

In [ ]:
plot_map_estimates(nlme_surrogate)

In [ ]:
_ = check_surrogate_validity_gp(nlme_surrogate)